In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
# Split Based on income
df["income_cat"] = pd.cut(df["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5])

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split

strat_train, strat_test = train_test_split(df, test_size=0.2, random_state=556,
                               stratify=df['income_cat'])


housing = strat_train.drop('income_cat', axis=1)
housing_labels = strat_train['median_house_value'].copy()
housing = housing.drop("median_house_value", axis=1)

# fix test set
housing_test = strat_test.drop('income_cat', axis=1)
housing_labels_test = strat_test['median_house_value'].copy()
housing_test = housing_test.drop("median_house_value", axis=1)


print(f"Train shape: {housing.shape}")
print(f"Train labels shape: {housing_labels.shape}")

print(f"Test shape: {housing_test.shape}")
print(f"Test labels shape: {housing_labels_test.shape}")

Train shape: (16512, 9)
Train labels shape: (16512,)
Test shape: (4128, 9)
Test labels shape: (4128,)


In [4]:
numerical_features = housing.select_dtypes(include=[np.number]).columns
categorical_features = housing.select_dtypes(include='object').columns

print(f"Numerical features: {numerical_features}")
print(f"Categorical features: {categorical_features}")

Numerical features: Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income'],
      dtype='object')
Categorical features: Index(['ocean_proximity'], dtype='object')


In [7]:
# Numerical pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ('num_imputer', SimpleImputer(strategy='median')),
    ('std_scaler', StandardScaler())
])

In [9]:
# Categorical pipeline
from sklearn.preprocessing import OneHotEncoder

cat_pipeline = Pipeline([
    ('cat_imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder())
])

In [25]:
categorical_features

Index(['ocean_proximity'], dtype='object')

In [10]:
# Full pipeline
from sklearn.compose import ColumnTransformer

full_pipeline = ColumnTransformer(transformers=[
    ('num', num_pipeline, numerical_features),
    ('cat', cat_pipeline, categorical_features)
])

In [11]:
full_pipeline

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('num_imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('std_scaler',
                                                  StandardScaler())]),
                                 Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('cat_imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('one_hot', OneHotEncoder())]),
                                 Index(['ocean_proximity'], dtype='object'))])

In [23]:
# Random Forest Regressor

from sklearn.ensemble import RandomForestRegressor
# import linear model
from sklearn.linear_model import LinearRegression

# forest_reg = RandomForestRegressor(n_estimators=100, random_state=42)

lin_reg = LinearRegression()



model = Pipeline([
    ('preprocessing', full_pipeline),
    # ('forest_reg_model', forest_reg)
    ('lin_reg_model', lin_reg)
])

model

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('num_imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('std_scaler',
                                                                   StandardScaler())]),
                                                  Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('cat_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('one_hot',
                                                                   OneHotEncoder())]),
                                                  Index(['ocean_proximity'], dtype='object'))])),
                ('lin_reg_model', LinearRegression())])

In [15]:
model.fit(housing, housing_labels)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('num_imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('std_scaler',
                                                                   StandardScaler())]),
                                                  Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('cat_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('one_hot',
                                                                   OneHotEncoder())]),
                                                  Index(['ocean_proximity'], dtype='object'))])),
                ('forest_reg_model', RandomForestRegressor(random_state=42))])

In [17]:
# Evaluate the model
from sklearn.metrics import root_mean_squared_error as rmse

housing_predictions = model.predict(housing_test)
forest_mse = rmse(housing_labels_test, housing_predictions)
forest_rmse = np.sqrt(forest_mse)
forest_rmse

221.6754824172276

In [18]:
model.score(housing_test, housing_labels_test)

0.8158253139121373

In [19]:
# save the model
import joblib
joblib.dump(model, 'random_forest_model.pkl')

['random_forest_model.pkl']

In [26]:
import pandas as pd
import numpy as np
import joblib

example = {
    "latitude": 37.88,
    "longitude": -122.23,
    "housing_median_age": 41.0,
    "total_rooms": 880.0,
    "total_bedrooms": 129.0,
    "population": 322.0,
    "households": 126.0,
    "median_income": 8.3252,
    "ocean_proximity": "NEAR BAY"
}

example_df = pd.DataFrame([example])
model = joblib.load('random_forest_model.pkl')
result = model.predict(example_df)
print(result)


[444442.13]
